In [1]:
import json
import zipfile
from collections import Counter
from datetime import datetime, timezone, timedelta
from pathlib import Path
from xml.etree import ElementTree as ET

ROOT = Path("/home/pasha/projects/bybit-history-downloader")

ARCHIVES = {
    "SPOT_BTC": ROOT / "data/raw/bitget/spot/spot/order_book_l2/BTCUSDT/USDT-BTCUSDT_1_20260901.zip",
    "SPOT_ETH": ROOT / "data/raw/bitget/spot/spot/order_book_l2/ETHUSDC/USDC-ETHUSDC_1_20260901.zip",
    "LINEAR": ROOT / "data/raw/bitget/perpetual/linear/order_book_l2/BTCUSDT/BTCUSDT-BTCUSDT_3_20260901.zip",
}

NS = {"x": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}

UTC8 = timezone(timedelta(hours=8))


def read_snapshots(archive_path):
    with zipfile.ZipFile(archive_path) as outer:
        members = [
            name for name in outer.namelist()
            if name.lower().endswith(".xlsx")
        ]

        if len(members) != 1:
            raise ValueError(f"Expected one XLSX workbook, found: {members}")

        with outer.open(members[0]) as workbook_stream:
            with zipfile.ZipFile(workbook_stream) as workbook:
                sheets = [
                    name for name in workbook.namelist()
                    if name.startswith("xl/worksheets/sheet")
                    and name.endswith(".xml")
                ]

                if len(sheets) != 1:
                    raise ValueError(f"Expected one worksheet, found: {sheets}")

                with workbook.open(sheets[0]) as sheet:
                    context = ET.iterparse(sheet, events=("end",))

                    for _, element in context:
                        if element.tag != f"{{{NS['x']}}}row":
                            continue

                        values = {}

                        for cell in element.findall("x:c", NS):
                            ref = cell.attrib.get("r", "")
                            column = "".join(
                                char for char in ref if char.isalpha()
                            )

                            value = cell.find("x:v", NS)

                            if value is None:
                                inline = cell.find("x:is/x:t", NS)
                                text = inline.text if inline is not None else None
                            else:
                                text = value.text

                            values[column] = text

                        element.clear()

                        if values.get("A") == "timestamp":
                            continue

                        if not all(values.get(col) for col in ("A", "B", "C")):
                            continue

                        yield {
                            "timestamp": int(values["A"]),
                            "asks": json.loads(values["B"]),
                            "bids": json.loads(values["C"]),
                        }


def analyze_market(name, archive_path):
    print("\n" + "=" * 90)
    print(f"MARKET: {name}")
    print("=" * 90)

    timestamps = []
    depths = Counter()
    intervals = Counter()

    violations = Counter()
    examples = {}

    total = 0
    previous_timestamp = None

    for snapshot in read_snapshots(archive_path):
        total += 1

        timestamp = snapshot["timestamp"]
        asks = snapshot["asks"]
        bids = snapshot["bids"]

        timestamps.append(timestamp)
        depths[(len(bids), len(asks))] += 1

        if previous_timestamp is not None:
            delta = timestamp - previous_timestamp

            if delta < 0:
                violations["timestamp_decrease"] += 1
            elif delta == 0:
                violations["duplicate_adjacent_timestamp"] += 1
            else:
                intervals[delta] += 1

        previous_timestamp = timestamp

        if not bids or not asks:
            violations["empty_book_side"] += 1
            continue

        bid_prices = [level[0] for level in bids]
        ask_prices = [level[0] for level in asks]

        if any(
            bid_prices[i] <= bid_prices[i + 1]
            for i in range(len(bid_prices) - 1)
        ):
            violations["bid_ordering"] += 1

        if any(
            ask_prices[i] >= ask_prices[i + 1]
            for i in range(len(ask_prices) - 1)
        ):
            violations["ask_ordering"] += 1

        if bid_prices[0] >= ask_prices[0]:
            violations["crossed_book"] += 1

            examples.setdefault(
                "crossed_book",
                {
                    "timestamp": timestamp,
                    "best_bid": bid_prices[0],
                    "best_ask": ask_prices[0],
                },
            )

        for side, levels in (("bid", bids), ("ask", asks)):
            for price, quantity in levels:
                if price <= 0:
                    violations["invalid_price"] += 1

                if quantity <= 0:
                    violations["invalid_quantity"] += 1

    if not timestamps:
        raise ValueError(f"No snapshots found: {name}")

    unique_timestamps = len(set(timestamps))

    start = min(timestamps)
    end = max(timestamps)

    print(f"Snapshots              : {total}")
    print(f"Unique timestamps      : {unique_timestamps}")
    print(f"Duplicate timestamps   : {total - unique_timestamps}")

    print("\nTIMESTAMP RANGE")

    print(
        "First UTC              :",
        datetime.fromtimestamp(start, timezone.utc),
    )

    print(
        "Last UTC               :",
        datetime.fromtimestamp(end, timezone.utc),
    )

    print(
        "First UTC+8            :",
        datetime.fromtimestamp(start, UTC8),
    )

    print(
        "Last UTC+8             :",
        datetime.fromtimestamp(end, UTC8),
    )

    print("\nSNAPSHOT DEPTH")

    for (bids, asks), count in depths.most_common(10):
        print(f"  bids={bids:4} asks={asks:4} count={count}")

    sorted_timestamps = sorted(timestamps)

    sorted_intervals = Counter(
        b - a
        for a, b in zip(
            sorted_timestamps,
            sorted_timestamps[1:],
        )
    )

    print("\nSORTED SNAPSHOT INTERVALS")

    for interval, count in sorted_intervals.most_common(10):
        print(f"  interval={interval:5}s count={count}")

    print("\nSOURCE ORDERING")

    print(
        "Chronological          :",
        timestamps == sorted_timestamps,
    )

    print("\nINTEGRITY VIOLATIONS")

    if violations:
        for key, count in violations.items():
            print(f"  {key}: {count}")
    else:
        print("  NONE")

    if examples:
        print("\nEXAMPLES")
        print(json.dumps(examples, indent=2))

    print("\nRESULT: ANALYSIS COMPLETE")


for name, archive_path in ARCHIVES.items():
    analyze_market(name, archive_path)


MARKET: SPOT_BTC
Snapshots              : 4297
Unique timestamps      : 4296
Duplicate timestamps   : 1

TIMESTAMP RANGE
First UTC              : 2026-08-31 16:00:23+00:00
Last UTC               : 2026-09-01 15:58:23+00:00
First UTC+8            : 2026-09-01 00:00:23+08:00
Last UTC+8             : 2026-09-01 23:58:23+08:00

SNAPSHOT DEPTH
  bids= 500 asks= 500 count=4297

SORTED SNAPSHOT INTERVALS
  interval=   20s count=3533
  interval=   19s count=67
  interval=   22s count=62
  interval=   21s count=58
  interval=   18s count=51
  interval=   23s count=47
  interval=   24s count=44
  interval=   25s count=43
  interval=   14s count=43
  interval=   16s count=37

SOURCE ORDERING
Chronological          : False

INTEGRITY VIOLATIONS
  timestamp_decrease: 2116

RESULT: ANALYSIS COMPLETE

MARKET: SPOT_ETH
Snapshots              : 4297
Unique timestamps      : 4296
Duplicate timestamps   : 1

TIMESTAMP RANGE
First UTC              : 2026-08-31 16:00:23+00:00
Last UTC               : 2026

In [2]:
import json
import zipfile
from decimal import Decimal, getcontext
from pathlib import Path

getcontext().prec = 40

ROOT = Path("/home/pasha/projects/bybit-history-downloader")

JOBS = {
    "spot_btc": "bitget-spot-spot-BTCUSDT-l2-20260901-20260904-d110.json",
    "spot_eth": "bitget-spot-spot-ETHUSDC-l2-20260901-20260904-d117.json",
    "linear": "bitget-perpetual-linear-BTCUSDT-l2-20260901-20260904-d94.json",
}

for market, filename in JOBS.items():
    job_path = ROOT / "data/.jobs/process" / filename

    task = json.loads(job_path.read_text())["tasks"][0]

    archive_path = Path(task["input_path"])

    # Reuse the read_snapshots() helper from Stage 28.34.3.
    # It yields dictionaries containing timestamp, asks, and bids.
    first = min(
        enumerate(read_snapshots(archive_path), start=1),
        key=lambda item: (item[1]["timestamp"], item[0]),
    )

    ordinal, snapshot = first

    print("\n" + "=" * 90)
    print("MARKET:", market.upper())
    print("SOURCE ORDINAL:", ordinal)
    print("TIMESTAMP:", snapshot["timestamp"])
    print("=" * 90)

    samples = []

    for side in ("bids", "asks"):
        for price_raw, quantity_raw in snapshot[side][:3]:
            price = Decimal(str(price_raw))
            quantity = Decimal(str(quantity_raw))

            samples.append({
                "side": "bid" if side == "bids" else "ask",
                "price": str(price),
                "source_quantity": str(quantity),
                "expected_base": str(quantity),
                "expected_quote": str(quantity * price),
                "expected_contracts": None,
            })

    print(json.dumps(samples, indent=2))


MARKET: SPOT_BTC
SOURCE ORDINAL: 1661
TIMESTAMP: 1788192023
[
  {
    "side": "bid",
    "price": "78557.69",
    "source_quantity": "1.037168",
    "expected_base": "1.037168",
    "expected_quote": "81477.52222192",
    "expected_contracts": null
  },
  {
    "side": "bid",
    "price": "78557.63",
    "source_quantity": "0.06",
    "expected_base": "0.06",
    "expected_quote": "4713.4578",
    "expected_contracts": null
  },
  {
    "side": "bid",
    "price": "78557.62",
    "source_quantity": "0.000019",
    "expected_base": "0.000019",
    "expected_quote": "1.49259478",
    "expected_contracts": null
  },
  {
    "side": "ask",
    "price": "78557.7",
    "source_quantity": "0.239515",
    "expected_base": "0.239515",
    "expected_quote": "18815.7475155",
    "expected_contracts": null
  },
  {
    "side": "ask",
    "price": "78559.75",
    "source_quantity": "0.005759",
    "expected_base": "0.005759",
    "expected_quote": "452.42560025",
    "expected_contracts": null
  }